In [ ]:
from pathlib import Path

import iplotx as ipx
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn as sns
from sklearn.decomposition import PCA
from statsmodels.multivariate.factor import Factor

from climate_attitudes import configure_mpl
from climate_attitudes.correlation import Correlation
from climate_attitudes.dataset import Dataset
from climate_attitudes.datasets.imputed_reduced import GROUPS as column_groups
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import plot_corr_with_dendro

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rc("figure", dpi=150)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="ds1_5")

In [ ]:
indices = dataset.indices.collect()
indices = indices.with_columns(
    (cs.exclude("participant_id", "wave") - cs.exclude("participant_id", "wave").mean())
    / cs.exclude("participant_id", "wave").std()
)
indices

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="ds1_5")
resp = dataset.response.collect()
new_resp = resp.clone()


pcas = {}
for group_name, columns in column_groups.items():
    X = resp.select(*columns).to_numpy()
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    pca = PCA(n_components=1)
    pca.fit(X)
    efa = Factor(X, n_factor=1).fit()
    pcas[group_name] = efa
    X_reduced = pca.transform(X).flatten()
    X_reduced = efa.factor_scoring().flatten()
    new_resp = new_resp.with_columns(
        pl.Series(X_reduced).alias(group_name.lower().replace(" ", "_"))
    ).drop(*columns)


dataset.response = new_resp.lazy()

In [ ]:
group_name = "Politics"

In [ ]:
for group_name in column_groups:
    labels = resp.select(*column_groups[group_name]).columns

    pca = pcas[group_name]

    fig, ax = plt.subplots(figsize=(3, 1.5), constrained_layout=True)

    cmap = sns.diverging_palette(20, 230, as_cmap=True)

    sns.heatmap(
        # (pca.components_.T * np.sqrt(pca.explained_variance_)).T,
        pca.loadings.T,
        center=0,
        annot=True,
        fmt=".1f",
        linewidths=0.5,
        square=True,
        cmap=cmap,
        # cbar_kws={"aspect": },
        ax=ax,
    )
    print(group_name)
    print(labels)

    ax.set_xticks(
        np.arange(pca.loadings.shape[0]) + 0.5,
        labels,
        rotation=45,
        horizontalalignment="right",
    )
    ax.set_title(f"PCA Factor Loading ({group_name})")

    plt.show()

In [ ]:
efa.loadings.shape

In [ ]:
dataset_std = dataset.standardise(cs.exclude("participant_id", "wave"))
resp_std = dataset_std.response.collect()

In [ ]:
plot_corr_with_dendro(indices, kind=Correlation.PARTIAL_GLASSO)

In [ ]:
def plot_corr_network(df, corr, threshold: float = 0.05, directed: bool = False):
    fig, ax = plt.subplots(figsize=(15, 15), constrained_layout=True)

    DIVERGING_CMAP = sns.diverging_palette(20, 230, as_cmap=True)

    # Generate a mask for the upper triangle
    if not directed:
        mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    else:
        mask = np.full_like(corr, fill_value=False, dtype=bool)

    # If `mask_below` set, reset square colour below abs value to zero
    mask[abs(corr) < threshold] = True

    # Remove questions where only diagonal is unmasked
    keep_idxes = (2 * mask.shape[1] - (mask.sum(axis=1) + mask.sum(axis=0))) > 2
    corr = corr[keep_idxes][:, keep_idxes]
    mask = mask[keep_idxes][:, keep_idxes]
    node_labels = np.asarray(df.columns)[keep_idxes]

    # ======== Network
    adj = corr
    adj[np.diag_indices_from(adj)] = 0
    adj[mask] = 0
    if directed:
        G = nx.from_numpy_array(adj, create_using=nx.DiGraph)
    else:
        G = nx.from_numpy_array(adj)
    edge_linewidths = {(u, v): z["weight"] * 12 for u, v, z in G.edges(data=True)}
    edge_colours = [z["weight"] for u, v, z in G.edges(data=True)]
    edge_labels = [f"{z['weight']:.2f}" for u, v, z in G.edges(data=True)]
    layout = nx.forceatlas2_layout(G, gravity=1)

    with ipx.style.context(
        [
            "hollow",
            {
                "vertex": {
                    "linewidth": 1,
                },
                "edge": {
                    "color": edge_colours,
                    "alpha": 1,
                    "cmap": DIVERGING_CMAP,
                    "norm": mcolors.Normalize(vmin=-1, vmax=1),
                },
            },
        ]
    ):
        network_artist = ipx.network(
            G,
            layout=layout,
            tension=1,
            edge_labels=edge_labels,
            node_labels=node_labels,
            edge_linewidth=edge_linewidths,
            edge_curved=True,
            # aspect="equal",
            margins=0.1,
            edge_label_bbox=dict(
                edgecolor="black",
                facecolor="white",
                linewidth=0.25,
                boxstyle="round,pad=0.3",
            ),
            edge_label_rotate=True,
            vertex_facecolor="white",
            vertex_zorder=3,
            ax=ax,
        )[0]
        fig.colorbar(
            network_artist.get_edges(),
            shrink=0.6,
            aspect=30,
            ax=ax,
        )

    fig.suptitle("Partial correlation")

In [ ]:
corr = Correlation.PARTIAL_GLASSO.calculate(resp_std, assume_centered=True)
plot_corr_network(resp_std.drop("participant_id", "wave"), corr)

In [ ]:
corr = Correlation.PARTIAL_GLASSO.calculate(resp_std, assume_centered=True)
plot_corr_network(resp_std.drop("participant_id", "wave"), corr)

In [ ]:
sns.displot(resp_std, x="politics")

In [ ]:
sns.displot(resp_std, x="extreme_weather")

In [ ]:
sns.displot(resp_std, x="cc_behaviour_change")

In [ ]:
sns.displot(resp_std, x="cc_rational")

In [ ]:
sns.displot(resp_std, x="cc_impacts")

In [ ]:
sns.displot(resp_std, x="cc_policy")